# LeetCode 1599 — Maximum Profit of Operating a Centennial Wheel bằng Python

Link đề: https://leetcode.com/problems/maximum-profit-of-operating-a-centennial-wheel/ · Độ khó: **Medium** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def minOperationsMaxProfit(self, customers: List[int], boardingCost: int, runningCost: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ một đề **mô phỏng (simulation)** dài: tách được **trạng thái** (số người chờ, số người đã lên, số lượt quay, lợi nhuận) và **luật chuyển** mỗi lượt quay.
2. Viết baseline "thử mọi số lượt quay `k`, mô phỏng lại từ đầu" để hiểu đúng câu hỏi "**số lượt ít nhất** để lợi nhuận **lớn nhất**".
3. Viết lời giải một lượt quét: mô phỏng từng vòng, cập nhật lợi nhuận lớn nhất bằng so sánh **nghiêm ngặt `>`** để giữ số lượt **sớm nhất**.
4. Tối ưu "tua nhanh" phần cuối khi không còn khách mới (chia lấy nguyên `// 4`), và nhận xét cắt sớm khi `4·boardingCost ≤ runningCost`.
5. Xử lý các chi tiết: vẫn phải quay khi không có ai chờ nhưng khách sẽ đến sau, lợi nhuận phải **dương** mới hợp lệ (trả `-1`).

## 2. Đề bài & ví dụ

Bạn vận hành một vòng quay (Centennial Wheel) có 4 cabin, mỗi cabin chở **tối đa 4 người**. Mỗi lần quay (ngược chiều kim đồng hồ) tốn `runningCost` đô.

Cho mảng `customers` độ dài `n`: `customers[i]` là số khách mới đến **ngay trước lượt quay thứ `i`** (đánh số từ 0) — tức là phải quay `i` lần rồi khách `customers[i]` mới tới. **Không được để khách chờ nếu cabin còn chỗ.** Mỗi khách trả `boardingCost` đô khi lên cabin ở sát mặt đất, và xuống khi cabin đó trở lại mặt đất.

Bạn có thể dừng phục vụ bất cứ lúc nào (kể cả khi chưa phục vụ hết khách). Khi dừng, các lượt quay sau đó (để đưa khách xuống an toàn) là **miễn phí**. Nếu đang có hơn 4 người chờ thì chỉ 4 người lên, số còn lại chờ lượt sau.

Trả về **số lượt quay ít nhất** cần thực hiện để **lợi nhuận lớn nhất**. Nếu không có kịch bản nào lợi nhuận **dương**, trả `-1`.

**Ví dụ 1:** `customers = [8,3]`, `boardingCost = 5`, `runningCost = 6` → `3`. Lợi nhuận sau từng lượt: `14, 28, 37`.

**Ví dụ 2:** `customers = [10,9,6]`, `boardingCost = 6`, `runningCost = 4` → `7`. Lợi nhuận: `20, 40, 60, 80, 100, 120, 122`.

**Ví dụ 3:** `customers = [3,4,0,5,1]`, `boardingCost = 1`, `runningCost = 92` → `-1`. Lợi nhuận luôn âm.

**Ràng buộc:**
- `n == customers.length`, `1 ≤ n ≤ 10⁵`
- `0 ≤ customers[i] ≤ 50`
- `1 ≤ boardingCost, runningCost ≤ 100`

In [1]:
import random
import time
from typing import List

examples = [
    ([8, 3], 5, 6, 3),
    ([10, 9, 6], 6, 4, 7),
    ([3, 4, 0, 5, 1], 1, 92, -1),
]
for cus, b, r, want in examples:
    print(f"customers={cus}, boardingCost={b}, runningCost={r} -> mong đợi {want}")

customers=[8, 3], boardingCost=5, runningCost=6 -> mong đợi 3
customers=[10, 9, 6], boardingCost=6, runningCost=4 -> mong đợi 7
customers=[3, 4, 0, 5, 1], boardingCost=1, runningCost=92 -> mong đợi -1


## 3. Kiến thức nền

### 3.1 Mô phỏng bằng vòng `while` với nhiều điều kiện

Vòng quay tiếp tục khi **còn khách sắp đến** (`i < n`) **hoặc còn người chờ** (`waiting > 0`). Đây là mẫu "chạy cho tới khi mọi nguồn việc đều cạn".

In [2]:
arrivals = [2, 0, 5]
i, waiting, turns = 0, 0, 0
while i < len(arrivals) or waiting > 0:
    if i < len(arrivals):
        waiting += arrivals[i]; i += 1
    board = min(4, waiting); waiting -= board; turns += 1
    print(f"lượt {turns}: lên {board}, còn chờ {waiting}")

lượt 1: lên 2, còn chờ 0
lượt 2: lên 0, còn chờ 0
lượt 3: lên 4, còn chờ 1
lượt 4: lên 1, còn chờ 0


### 3.2 `min` để giới hạn sức chứa

`board = min(4, waiting)` — lên tối đa 4 người. Ví dụ `waiting = 9` ⇒ lên 4; `waiting = 3` ⇒ lên 3.

In [3]:
for w in [0, 3, 4, 9]:
    print(w, "chờ -> lên", min(4, w))

0 chờ -> lên 0
3 chờ -> lên 3
4 chờ -> lên 4
9 chờ -> lên 4


### 3.3 Theo dõi "giá trị lớn nhất và thời điểm sớm nhất"

Dùng `>` (nghiêm ngặt) khi cập nhật: nếu giá trị mới chỉ **bằng** giá trị tốt nhất thì giữ thời điểm **cũ** (sớm hơn). Nếu dùng `>=` sẽ lấy thời điểm **muộn nhất**.

In [4]:
vals = [3, 7, 5, 7, 2]
best, at_strict, at_ge = float("-inf"), -1, -1
best2 = float("-inf")
for t, v in enumerate(vals, 1):
    if v > best: best, at_strict = v, t
    if v >= best2: best2, at_ge = v, t
print("max =", best, "| sớm nhất (>):", at_strict, "| muộn nhất (>=):", at_ge)

max = 7 | sớm nhất (>): 2 | muộn nhất (>=): 4


### 3.4 `divmod` để "tua nhanh"

`divmod(W, 4)` trả `(W // 4, W % 4)`: số lượt chở đủ 4 người và số người lẻ còn lại.

In [5]:
print(divmod(13, 4), divmod(8, 4), divmod(3, 4))

(3, 1) (2, 0) (0, 3)


## 4. Ý tưởng nền tảng / trực giác

Lợi nhuận sau `k` lượt quay = `boardingCost × (tổng số người đã lên trong k lượt) − runningCost × k`.

Tính tay ví dụ 1 (`[8,3]`, `b = 5`, `r = 6`):

| lượt | khách đến | chờ trước khi lên | lên | chờ sau | tổng đã lên | lợi nhuận |
|---|---|---|---|---|---|---|
| 1 | 8 | 8 | 4 | 4 | 4 | 4·5 − 1·6 = 14 |
| 2 | 3 | 7 | 4 | 3 | 8 | 8·5 − 2·6 = 28 |
| 3 | — | 3 | 3 | 0 | 11 | 11·5 − 3·6 = 37 |

Lợi nhuận lớn nhất 37 đạt ở lượt 3 ⇒ trả 3.

**Quan sát:** mỗi lượt quay thay đổi lợi nhuận một lượng `board · b − r`. Lợi nhuận tăng khi `board · b > r` và giảm khi ngược lại. Ta không chỉ quan tâm lợi nhuận **cuối cùng** mà là **đỉnh** của dãy lợi nhuận.

In [6]:
def profits_list(customers, b, r):
    """Danh sách lợi nhuận sau mỗi lượt quay khi phục vụ đến hết khách."""
    res, i, waiting, boarded, turns = [], 0, 0, 0, 0
    while i < len(customers) or waiting > 0:
        if i < len(customers):
            waiting += customers[i]; i += 1
        board = min(4, waiting); waiting -= board
        boarded += board; turns += 1
        res.append(boarded * b - turns * r)
    return res

for cus, b, r, want in examples:
    print(cus, "->", profits_list(cus, b, r))

[8, 3] -> [14, 28, 37]
[10, 9, 6] -> [20, 40, 60, 80, 100, 120, 122]
[3, 4, 0, 5, 1] -> [-89, -177, -269, -357, -447]


## 5. Cách 1 — Brute Force / baseline: thử mọi số lượt `k`

**Ý tưởng:** "dừng sau đúng `k` lượt" là một kịch bản. Với mỗi `k = 1, 2, …, R` (R = số lượt để phục vụ hết khách), **mô phỏng lại từ đầu** `k` lượt để tính lợi nhuận, rồi chọn `k` nhỏ nhất có lợi nhuận lớn nhất (và dương).

Không cần xét `k > R`: sau khi hết khách, quay thêm chỉ tốn tiền.

In [7]:
def profit_after(customers, b, r, k):
    """Mô phỏng đúng k lượt quay rồi dừng; trả lợi nhuận."""
    i, waiting, boarded = 0, 0, 0
    for _ in range(k):
        if i < len(customers):
            waiting += customers[i]; i += 1
        board = min(4, waiting); waiting -= board; boarded += board
    return boarded * b - k * r

def total_rotations(customers):
    i, waiting, turns = 0, 0, 0
    while i < len(customers) or waiting > 0:
        if i < len(customers):
            waiting += customers[i]; i += 1
        waiting -= min(4, waiting); turns += 1
    return turns

def min_ops_brute(customers, b, r):
    best, ans = 0, -1                       # cần lợi nhuận > 0
    for k in range(1, total_rotations(customers) + 1):
        p = profit_after(customers, b, r, k)
        if p > best:
            best, ans = p, k
    return ans

for cus, b, r, want in examples:
    print(cus, "->", min_ops_brute(cus, b, r), "| mong đợi", want)

[8, 3] -> 3 | mong đợi 3
[10, 9, 6] -> 7 | mong đợi 7
[3, 4, 0, 5, 1] -> -1 | mong đợi -1


**Độ phức tạp:** gọi `R` là tổng số lượt (tối đa khoảng `n + 50n/4`). Mỗi `k` mô phỏng O(k) ⇒ **O(R²)**. Với `n = 10⁵`, `R` có thể ~`1.35·10⁶` ⇒ `R² ~ 10¹²` — không chạy nổi.

**Nút thắt:** mô phỏng lại từ đầu cho mỗi `k` là lãng phí: lợi nhuận sau `k+1` lượt chỉ là lợi nhuận sau `k` lượt cộng thêm `board·b − r`.

In [8]:
random.seed(0)
for n in [100, 200, 400]:
    cus = [random.randint(0, 50) for _ in range(n)]
    t = time.perf_counter(); min_ops_brute(cus, 5, 4)
    print(f"n={n}: R={total_rotations(cus)}, {time.perf_counter() - t:.3f}s")

n=100: R=625, 0.016s
n=200: R=1233, 0.063s


n=400: R=2438, 0.308s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** chỉ cần mô phỏng **một lần** từ đầu đến khi hết khách, sau mỗi lượt cập nhật `profit += board·b − r` và so sánh với `best`.

**Quy tắc:**
- Nếu `i < n` thì nhận khách `customers[i]` vào hàng chờ.
- Lên `board = min(4, waiting)`.
- `profit` mới = `boarded·b − turns·r`.
- **Nếu `profit > best`** (nghiêm ngặt) thì `best = profit`, `ans = turns`.
- Khởi tạo `best = 0`, `ans = -1`: chỉ lợi nhuận **dương** mới làm `ans` thay đổi ⇒ tự động xử lý "không có kịch bản dương ⇒ -1".

**Invariant:** sau lượt thứ `t`, `best` là lợi nhuận lớn nhất (so với 0) trong các lượt `1..t`, và `ans` là lượt **sớm nhất** đạt `best` (hoặc `-1`).

**Nhận xét cắt sớm:** mỗi lượt lên tối đa 4 người ⇒ lợi nhuận tăng tối đa `4b − r` mỗi lượt. Nếu `4b ≤ r` thì lợi nhuận không bao giờ dương ⇒ trả `-1` ngay.

**Tua nhanh:** sau khi khách cuối cùng đã đến (`i == n`), không còn gì bất ngờ: mỗi lượt lên đủ 4 người cho tới khi còn `< 4`. Nếu `4b > r` thì mọi lượt đủ 4 người đều có lời ⇒ làm hết chúng một lúc; lượt lẻ cuối (`rem` người) chỉ đáng làm nếu `rem·b > r`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng một lượt, theo dõi đỉnh lợi nhuận

Từng đoạn:
1. `i, waiting, boarded, turns = 0, 0, 0, 0`; `best, ans = 0, -1`.
2. Vòng `while i < n or waiting > 0`.
3. Nhận khách nếu còn; lên `min(4, waiting)`.
4. Cập nhật `turns`, `boarded`, tính `profit`, so với `best` bằng `>`.

In [9]:
def min_ops_simulate(customers, b, r):
    n = len(customers)
    i = waiting = boarded = turns = 0
    best, ans = 0, -1
    while i < n or waiting > 0:
        if i < n:
            waiting += customers[i]
            i += 1
        board = min(4, waiting)
        waiting -= board
        boarded += board
        turns += 1
        profit = boarded * b - turns * r
        if profit > best:
            best, ans = profit, turns
    return ans

for cus, b, r, want in examples:
    print(cus, "->", min_ops_simulate(cus, b, r), "| mong đợi", want)

[8, 3] -> 3 | mong đợi 3
[10, 9, 6] -> 7 | mong đợi 7
[3, 4, 0, 5, 1] -> -1 | mong đợi -1


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def minOperationsMaxProfit(self, customers: List[int], boardingCost: int, runningCost: int) -> int:
        n = len(customers)
        i = waiting = boarded = turns = 0
        best, ans = 0, -1
        while i < n or waiting > 0:
            if i < n:
                waiting += customers[i]
                i += 1
            board = min(4, waiting)
            waiting -= board
            boarded += board
            turns += 1
            profit = boarded * boardingCost - turns * runningCost
            if profit > best:
                best, ans = profit, turns
        return ans
```

**Độ phức tạp:** Time **O(R)** với `R ≤ n + ⌈Σcustomers / 4⌉` (gợi ý của đề: không quá `50/4 · n`), Space O(1).

### 7.2 Cách 3 — Mô phỏng phần có khách đến + tua nhanh phần đuôi + cắt sớm

1. Nếu `4b ≤ r` ⇒ `-1` ngay.
2. Mô phỏng như Cách 2 **chỉ trong `n` lượt đầu** (khi còn khách đến).
3. Sau đó còn `W = waiting` người: `q, rem = divmod(W, 4)`. Vì `4b > r`, mỗi lượt trong `q` lượt đủ người đều tăng lợi nhuận ⇒ đỉnh mới (nếu có) ở cuối: `profit + q·(4b − r)` tại lượt `n + q`.
4. Lượt lẻ: nếu `rem > 0` và `rem·b > r` thì thêm một lượt nữa có lời.

Mỗi bước tua chỉ cập nhật `best/ans` nếu **lớn hơn hẳn** — vẫn đúng logic "sớm nhất".

In [10]:
def min_ops_fast(customers, b, r):
    if 4 * b <= r:
        return -1
    waiting = boarded = turns = 0
    best, ans = 0, -1
    for c in customers:                      # n lượt đầu, mỗi lượt có (có thể 0) khách đến
        waiting += c
        board = min(4, waiting)
        waiting -= board
        boarded += board
        turns += 1
        profit = boarded * b - turns * r
        if profit > best:
            best, ans = profit, turns
    profit = boarded * b - turns * r
    q, rem = divmod(waiting, 4)
    if q > 0:                                # q lượt đủ 4 người, mỗi lượt +(4b - r) > 0
        profit += q * (4 * b - r)
        turns += q
        if profit > best:
            best, ans = profit, turns
    if rem > 0 and rem * b > r:              # lượt lẻ cuối chỉ làm nếu có lời
        profit += rem * b - r
        turns += 1
        if profit > best:
            best, ans = profit, turns
    return ans

for cus, b, r, want in examples:
    print(cus, "->", min_ops_fast(cus, b, r), "| mong đợi", want)

[8, 3] -> 3 | mong đợi 3
[10, 9, 6] -> 7 | mong đợi 7
[3, 4, 0, 5, 1] -> -1 | mong đợi -1


Tại sao ở bước tua nhanh không cần lo "đỉnh nằm giữa q lượt"? Vì mỗi lượt đủ người cộng một lượng **dương** `4b − r`, dãy lợi nhuận **tăng nghiêm ngặt** trong đoạn đó ⇒ giá trị lớn nhất ở cuối đoạn.

Code nộp LeetCode của Cách 3:

```python
class Solution:
    def minOperationsMaxProfit(self, customers: List[int], boardingCost: int, runningCost: int) -> int:
        b, r = boardingCost, runningCost
        if 4 * b <= r:
            return -1
        waiting = boarded = turns = 0
        best, ans = 0, -1
        for c in customers:
            waiting += c
            board = min(4, waiting)
            waiting -= board
            boarded += board
            turns += 1
            profit = boarded * b - turns * r
            if profit > best:
                best, ans = profit, turns
        profit = boarded * b - turns * r
        q, rem = divmod(waiting, 4)
        if q > 0:
            profit += q * (4 * b - r)
            turns += q
            if profit > best:
                best, ans = profit, turns
        if rem > 0 and rem * b > r:
            profit += rem * b - r
            turns += 1
            if profit > best:
                best, ans = profit, turns
        return ans
```

**Độ phức tạp:** Time **O(n)** (không phụ thuộc số người chờ), Space O(1).

### 7.3 Cách 4 — Tính dãy lợi nhuận rồi lấy `argmax` (phong cách "dữ liệu")

Nếu thích tách bạch: sinh danh sách lợi nhuận `P[1..R]` (hàm `profits_list` ở mục 4), rồi `ans = 1 + P.index(max(P))` nếu `max(P) > 0`, ngược lại `-1`. `list.index` trả vị trí **đầu tiên** ⇒ đúng "sớm nhất". Tốn O(R) bộ nhớ, nhưng code rất dễ kiểm chứng.

In [11]:
def min_ops_argmax(customers, b, r):
    P = profits_list(customers, b, r)
    m = max(P)
    return P.index(m) + 1 if m > 0 else -1

for cus, b, r, want in examples:
    print(cus, "->", min_ops_argmax(cus, b, r), "| mong đợi", want)

[8, 3] -> 3 | mong đợi 3
[10, 9, 6] -> 7 | mong đợi 7
[3, 4, 0, 5, 1] -> -1 | mong đợi -1


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def minOperationsMaxProfit(self, customers: List[int], boardingCost: int, runningCost: int) -> int:
        P, i, waiting, boarded, turns = [], 0, 0, 0, 0
        while i < len(customers) or waiting > 0:
            if i < len(customers):
                waiting += customers[i]; i += 1
            board = min(4, waiting); waiting -= board
            boarded += board; turns += 1
            P.append(boarded * boardingCost - turns * runningCost)
        m = max(P)
        return P.index(m) + 1 if m > 0 else -1
```

**Độ phức tạp:** Time O(R), Space O(R).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 (`[10,9,6]`, `b = 6`, `r = 4`):

| lượt | đến | chờ trước | lên | chờ sau | đã lên | lợi nhuận | best / ans |
|---|---|---|---|---|---|---|---|
| 1 | 10 | 10 | 4 | 6 | 4 | 20 | 20 / 1 |
| 2 | 9 | 15 | 4 | 11 | 8 | 40 | 40 / 2 |
| 3 | 6 | 17 | 4 | 13 | 12 | 60 | 60 / 3 |
| 4 | — | 13 | 4 | 9 | 16 | 80 | 80 / 4 |
| 5 | — | 9 | 4 | 5 | 20 | 100 | 100 / 5 |
| 6 | — | 5 | 4 | 1 | 24 | 120 | 120 / 6 |
| 7 | — | 1 | 1 | 0 | 25 | 122 | 122 / 7 |

In [12]:
def trace_simulate(customers, b, r):
    n = len(customers)
    i = waiting = boarded = turns = 0
    best, ans = 0, -1
    print("lượt | đến | chờ trước | lên | chờ sau | đã lên | lợi nhuận | best/ans")
    while i < n or waiting > 0:
        arrive = "-"
        if i < n:
            arrive = customers[i]; waiting += customers[i]; i += 1
        before = waiting
        board = min(4, waiting); waiting -= board
        boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best:
            best, ans = profit, turns
        print(f"{turns:4d} | {arrive!s:>4} | {before:9d} | {board:3d} | {waiting:7d} | {boarded:6d} | {profit:9d} | {best}/{ans}")
    print("=> kết quả", ans)
    return ans

trace_simulate([10, 9, 6], 6, 4)

lượt | đến | chờ trước | lên | chờ sau | đã lên | lợi nhuận | best/ans
   1 |   10 |        10 |   4 |       6 |      4 |        20 | 20/1
   2 |    9 |        15 |   4 |      11 |      8 |        40 | 40/2
   3 |    6 |        17 |   4 |      13 |     12 |        60 | 60/3
   4 |    - |        13 |   4 |       9 |     16 |        80 | 80/4
   5 |    - |         9 |   4 |       5 |     20 |       100 | 100/5
   6 |    - |         5 |   4 |       1 |     24 |       120 | 120/6
   7 |    - |         1 |   1 |       0 |     25 |       122 | 122/7
=> kết quả 7


7

In [13]:
trace_simulate([3, 4, 0, 5, 1], 1, 92)

lượt | đến | chờ trước | lên | chờ sau | đã lên | lợi nhuận | best/ans
   1 |    3 |         3 |   3 |       0 |      3 |       -89 | 0/-1
   2 |    4 |         4 |   4 |       0 |      7 |      -177 | 0/-1
   3 |    0 |         0 |   0 |       0 |      7 |      -269 | 0/-1
   4 |    5 |         5 |   4 |       1 |     11 |      -357 | 0/-1
   5 |    1 |         2 |   2 |       0 |     13 |      -447 | 0/-1
=> kết quả -1


-1

Ví dụ 3: lợi nhuận luôn âm, `best` giữ 0 và `ans` giữ `-1`. Để ý lượt 3: không ai đến, không ai chờ, nhưng **vẫn phải quay** vì khách lượt 4 chỉ tới sau khi đã quay 3 lần.

In [14]:
# Trace Cách 3: phần tua nhanh
def trace_fast(customers, b, r):
    if 4 * b <= r:
        print("4b <= r -> -1 ngay"); return -1
    waiting = boarded = turns = 0; best, ans = 0, -1
    for c in customers:
        waiting += c; board = min(4, waiting); waiting -= board
        boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best: best, ans = profit, turns
    profit = boarded * b - turns * r
    print(f"hết khách mới sau {turns} lượt: lợi nhuận {profit}, còn chờ {waiting}, best/ans = {best}/{ans}")
    q, rem = divmod(waiting, 4)
    if q > 0:
        profit += q * (4 * b - r); turns += q
        if profit > best: best, ans = profit, turns
        print(f"tua {q} lượt đủ 4 người: lợi nhuận {profit} tại lượt {turns}, best/ans = {best}/{ans}")
    if rem > 0:
        if rem * b > r:
            profit += rem * b - r; turns += 1
            if profit > best: best, ans = profit, turns
            print(f"lượt lẻ {rem} người có lời: lợi nhuận {profit} tại lượt {turns}, best/ans = {best}/{ans}")
        else:
            print(f"lượt lẻ {rem} người: {rem}*{b} <= {r} -> không làm")
    print("=> kết quả", ans)
    return ans

trace_fast([10, 9, 6], 6, 4)
print()
trace_fast([10, 10, 6, 4, 7], 3, 8)

hết khách mới sau 3 lượt: lợi nhuận 60, còn chờ 13, best/ans = 60/3
tua 3 lượt đủ 4 người: lợi nhuận 120 tại lượt 6, best/ans = 120/6
lượt lẻ 1 người có lời: lợi nhuận 122 tại lượt 7, best/ans = 122/7
=> kết quả 7

hết khách mới sau 5 lượt: lợi nhuận 20, còn chờ 17, best/ans = 20/5
tua 4 lượt đủ 4 người: lợi nhuận 36 tại lượt 9, best/ans = 36/9
lượt lẻ 1 người: 1*3 <= 8 -> không làm
=> kết quả 9


9

## 9. Code nộp LeetCode

```python
from typing import List

class Solution:
    def minOperationsMaxProfit(self, customers: List[int], boardingCost: int, runningCost: int) -> int:
        n = len(customers)
        i = waiting = boarded = turns = 0
        best, ans = 0, -1                      # chỉ nhận lợi nhuận dương
        while i < n or waiting > 0:
            if i < n:
                waiting += customers[i]
                i += 1
            board = min(4, waiting)
            waiting -= board
            boarded += board
            turns += 1
            profit = boarded * boardingCost - turns * runningCost
            if profit > best:                  # '>' giữ số lượt sớm nhất
                best, ans = profit, turns
        return ans
```

In [15]:
class Solution:
    def minOperationsMaxProfit(self, customers: List[int], boardingCost: int, runningCost: int) -> int:
        n = len(customers)
        i = waiting = boarded = turns = 0
        best, ans = 0, -1
        while i < n or waiting > 0:
            if i < n:
                waiting += customers[i]
                i += 1
            board = min(4, waiting)
            waiting -= board
            boarded += board
            turns += 1
            profit = boarded * boardingCost - turns * runningCost
            if profit > best:
                best, ans = profit, turns
        return ans

sol = Solution()
for cus, b, r, want in examples:
    got = sol.minOperationsMaxProfit(cus, b, r)
    print(cus, "->", got, "OK" if got == want else "SAI")

[8, 3] -> 3 OK
[10, 9, 6] -> 7 OK
[3, 4, 0, 5, 1] -> -1 OK


## 10. Vì sao thuật toán đúng

1. **Không gian kịch bản:** mọi kịch bản là "dừng sau `k` lượt" với `k ≥ 1` (dừng ở `k = 0` cho lợi nhuận 0, không dương). Luật "không được để khách chờ khi còn chỗ" khiến số người lên ở mỗi lượt là **xác định** (`min(4, waiting)`), nên lợi nhuận sau `k` lượt là một hàm xác định `P(k)`.
2. **Chỉ cần xét `k ≤ R`:** sau lượt `R` (hết khách), mỗi lượt thêm giảm `r` ⇒ không thể tốt hơn.
3. **Mô phỏng một lần tính đúng mọi `P(k)`:** trạng thái sau `k` lượt của "dừng sau `k` lượt" trùng với trạng thái sau `k` lượt của mô phỏng đầy đủ (quyết định dừng không ảnh hưởng các lượt trước).
4. **Chọn đúng:** `best/ans` cập nhật khi `P(k) > best` ⇒ cuối cùng `best = max(0, max P)` và `ans` là `k` **nhỏ nhất** đạt max (vì giá trị bằng nhau về sau không ghi đè). Nếu `max P ≤ 0` thì `ans = -1`.
5. **Cách 3:** trong đoạn đuôi, `P` tăng nghiêm ngặt qua các lượt đủ 4 người (vì `4b − r > 0`), và lượt lẻ thay đổi `rem·b − r`; chỉ có hai "ứng viên đỉnh" là cuối đoạn đủ người và sau lượt lẻ (nếu có lời). Cắt sớm `4b ≤ r` đúng vì `P(k) ≤ k(4b − r) ≤ 0`.

## 11. Phân tích Time / Space

Gọi `n = len(customers)`, `S = Σ customers ≤ 50n`, `R` = tổng số lượt `≤ n + ⌈S/4⌉ ≤ 13.5n`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `k` | O(R²) | O(1) |
| 2. Mô phỏng một lượt | O(R) | O(1) |
| 3. Mô phỏng + tua nhanh | O(n) | O(1) |
| 4. Dãy lợi nhuận + argmax | O(R) | O(R) |

Đo trên input lớn nhất (`n = 10⁵`, mọi `customers[i] = 50`):

In [16]:
big = [50] * 10**5
for f in [min_ops_simulate, min_ops_fast, min_ops_argmax]:
    t = time.perf_counter(); res = f(big, 7, 20)
    print(f"{f.__name__:18s} -> {res}, {time.perf_counter() - t:.3f}s")

min_ops_simulate   -> 1250000, 0.223s
min_ops_fast       -> 1250000, 0.016s


min_ops_argmax     -> 1250000, 0.331s


## 12. Bộ test

Ví dụ đề, edge case (lợi nhuận đúng bằng 0, hòa lợi nhuận ở hai lượt, toàn 0 khách, khách đến sau nhiều lượt trống, `4b ≤ r`, `4b` vừa lớn hơn `r`), và đối chiếu ngẫu nhiên giữa các cách.

In [17]:
ALL = [min_ops_brute, min_ops_simulate, min_ops_fast, min_ops_argmax,
       lambda c, b, r: Solution().minOperationsMaxProfit(c, b, r)]
cases = examples + [
    ([1], 1, 1, -1),             # lợi nhuận 0 -> không dương
    ([0, 0, 0], 5, 1, -1),       # không ai đến
    ([0, 4], 5, 1, 2),           # lượt đầu trống vẫn phải quay
    ([4, 1], 5, 5, 1),           # hòa 15 ở lượt 1 và 2 -> lấy 1
    ([50], 1, 4, -1),            # 4b == r -> không bao giờ dương
    ([50], 2, 7, 12),            # 4b > r: 12 lượt đủ 4 người, lượt lẻ 2 người: 2*2 < 7
    ([2], 100, 1, 1),
    ([0, 0, 0, 0, 20], 10, 1, 9),
]
for cus, b, r, want in cases:
    for f in ALL:
        assert f(cus, b, r) == want, (f, cus, b, r, f(cus, b, r), want)
print("Ca cố định:", len(cases), "OK")

random.seed(2024)
for _ in range(1500):
    n = random.randint(1, 12)
    cus = [random.randint(0, 50) if random.random() < 0.7 else 0 for _ in range(n)]
    b, r = random.randint(1, 100), random.randint(1, 100)
    want = min_ops_brute(cus, b, r)
    for f in ALL[1:]:
        assert f(cus, b, r) == want, (f, cus, b, r)
print("Ngẫu nhiên nhỏ: OK")
for _ in range(30):
    n = random.randint(1, 3000)
    cus = [random.randint(0, 50) for _ in range(n)]
    b, r = random.randint(1, 100), random.randint(1, 100)
    assert len({f(cus, b, r) for f in ALL[1:]}) == 1
print("Ngẫu nhiên lớn (không brute): các cách đồng nhất")

Ca cố định: 11 OK
Ngẫu nhiên nhỏ: OK


Ngẫu nhiên lớn (không brute): các cách đồng nhất


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>=` khi cập nhật lợi nhuận lớn nhất

In [18]:
def wrong_ge(customers, b, r):
    n = len(customers); i = waiting = boarded = turns = 0; best, ans = 0, -1
    while i < n or waiting > 0:
        if i < n: waiting += customers[i]; i += 1
        board = min(4, waiting); waiting -= board; boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit >= best and profit > 0:      # SAI: '>=' lấy lượt muộn nhất
            best, ans = profit, turns
    return ans

args = ([4, 1], 5, 5)
print("sai:", wrong_ge(*args), "| đúng:", Solution().minOperationsMaxProfit(*args), "| dãy lợi nhuận:", profits_list(*args))

sai: 2 | đúng: 1 | dãy lợi nhuận: [15, 15]


**Vì sao sai:** lợi nhuận lượt 1 và lượt 2 cùng là 15 (lượt 2 lên 1 người, `1·5 − 5 = 0`). Đề hỏi số lượt **ít nhất** ⇒ 1. **Sửa:** dùng `>`.

### Lỗi 2 — Dừng khi hết khách mới đến (bỏ quên hàng chờ)

In [19]:
def wrong_stop_early(customers, b, r):
    waiting = boarded = turns = 0; best, ans = 0, -1
    for c in customers:                        # SAI: không quay tiếp khi còn người chờ
        waiting += c; board = min(4, waiting); waiting -= board
        boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best: best, ans = profit, turns
    return ans

args = ([10, 9, 6], 6, 4)
print("sai:", wrong_stop_early(*args), "| đúng:", Solution().minOperationsMaxProfit(*args))

sai: 3 | đúng: 7


**Vì sao sai:** sau 3 lượt vẫn còn 13 người chờ; phục vụ họ mang thêm lợi nhuận tới lượt 7. **Sửa:** điều kiện vòng lặp `while i < n or waiting > 0` (hoặc tua nhanh như Cách 3).

### Lỗi 3 — Dừng khi hàng chờ trống dù khách vẫn sẽ đến

In [20]:
def wrong_stop_when_empty(customers, b, r):
    n = len(customers); i = waiting = boarded = turns = 0; best, ans = 0, -1
    while i < n:
        waiting += customers[i]; i += 1
        if waiting == 0:                       # SAI: nghĩ rằng không có ai thì dừng
            break
        board = min(4, waiting); waiting -= board; boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best: best, ans = profit, turns
    while waiting > 0:
        board = min(4, waiting); waiting -= board; boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best: best, ans = profit, turns
    return ans

args = ([0, 4], 5, 1)
print("sai:", wrong_stop_when_empty(*args), "| đúng:", Solution().minOperationsMaxProfit(*args))

sai: -1 | đúng: 2


**Vì sao sai:** khách `customers[1]` chỉ đến **sau 1 lượt quay**. Phải quay lượt trống (lỗ `r`) để đón 4 khách có lời. **Sửa:** chỉ dừng khi **cả** `i == n` **và** `waiting == 0`.

### Lỗi 4 — Cho tất cả người chờ lên cùng lúc (quên sức chứa 4)

In [21]:
def wrong_no_cap(customers, b, r):
    n = len(customers); i = waiting = boarded = turns = 0; best, ans = 0, -1
    while i < n or waiting > 0:
        if i < n: waiting += customers[i]; i += 1
        board = waiting                         # SAI: không giới hạn 4
        waiting = 0; boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best: best, ans = profit, turns
    return ans

args = ([8, 3], 5, 6)
print("sai:", wrong_no_cap(*args), "| đúng:", Solution().minOperationsMaxProfit(*args))

sai: 2 | đúng: 3


**Vì sao sai:** mỗi cabin chỉ chở 4 người ⇒ 8 khách cần 2 lượt. **Sửa:** `board = min(4, waiting)`.

### Lỗi 5 — Coi lợi nhuận 0 là hợp lệ

In [22]:
def wrong_zero_ok(customers, b, r):
    n = len(customers); i = waiting = boarded = turns = 0; best, ans = -1, -1
    while i < n or waiting > 0:
        if i < n: waiting += customers[i]; i += 1
        board = min(4, waiting); waiting -= board; boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best: best, ans = profit, turns
    return ans if best >= 0 else -1             # SAI: đề yêu cầu lợi nhuận DƯƠNG

args = ([1], 1, 1)
print("sai:", wrong_zero_ok(*args), "| đúng:", Solution().minOperationsMaxProfit(*args))

sai: 1 | đúng: -1


**Vì sao sai:** "If there is no scenario where the profit is **positive**, return -1". Lợi nhuận 0 không dương. **Sửa:** khởi tạo `best = 0` và cập nhật bằng `>`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề mô phỏng dài ⇒ viết ra **trạng thái** và **luật mỗi bước** trước khi code; tự tính tay ví dụ của đề để chắc hiểu đúng (đề này có bảng giải thích chi tiết — hãy đối chiếu từng dòng).
- "Số bước **ít nhất** để đạt **max**" ⇒ cập nhật bằng `>` khi duyệt theo thời gian tăng dần.
- "Return -1 nếu không dương" ⇒ khởi tạo `best = 0, ans = -1`.
- Vòng lặp mô phỏng có hai nguồn việc (khách sắp đến, người đang chờ) ⇒ `while nguồn1 or nguồn2`.
- Phần "đuôi" có quy luật đều ⇒ nghĩ tới `divmod` để tua nhanh.

## 15. Biến thể của bài

1. **Sức chứa / số cabin khác:** thay `4` bằng tham số `cap` (bài tập 4).
2. **Chi phí quay thay đổi theo thời gian** (`runningCost[i]`).
3. **Trả về lợi nhuận lớn nhất** thay vì số lượt.
4. **Cho phép từ chối khách** (không bắt buộc cho lên): khi đó có thể có chiến lược khác — bài trở thành tối ưu hóa thật sự.
5. **LC 1701 — Average Waiting Time**, **LC 2073 — Time Needed to Buy Tickets**: các bài mô phỏng hàng đợi tương tự.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `k` | mô phỏng lại `k` lượt cho mỗi `k` | O(R²) | O(1) | hiểu đề, đối chiếu |
| 2. Mô phỏng một lượt | cộng dồn lợi nhuận, giữ đỉnh sớm nhất | O(R) | O(1) | **nộp bài** — ngắn, rõ |
| 3. Tua nhanh + cắt sớm | `divmod` cho phần đuôi, `4b ≤ r ⇒ -1` | O(n) | O(1) | khi số người chờ rất lớn |
| 4. Dãy + argmax | sinh dãy lợi nhuận, `index(max)` | O(R) | O(R) | dễ kiểm chứng, debug |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[4]`, `b=1`, `r=3`; (b) `[4]`, `b=1`, `r=4`; (c) `[0,0,8]`, `b=3`, `r=2`; (d) `[5]`, `b=10`, `r=9`.

**Bài 2 (tính tay):** Lập bảng từng lượt cho `customers = [5,5]`, `b = 3`, `r = 5` và cho đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `max_profit_value(customers, b, r)` trả về **lợi nhuận lớn nhất** (hoặc `0` nếu không có lợi nhuận dương).

**Bài 4 (tự viết lại):** Viết `min_ops_cap(customers, b, r, cap)` với sức chứa mỗi cabin là `cap` thay vì 4.

**Bài 5 (mini challenge):** Viết `min_ops_varcost(customers, b, costs)` trong đó lượt quay thứ `t` (từ 1) tốn `costs[(t-1) % len(costs)]` đô (chi phí lặp theo chu kỳ).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_profit_value(customers, b, r):
    n = len(customers)
    i = waiting = boarded = turns = 0
    best = 0
    while i < n or waiting > 0:
        # TODO: nhận khách, cho lên min(4, waiting), cập nhật turns/boarded
        # TODO: best = max(best, boarded * b - turns * r)
        pass
    return best

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_ops_cap(customers, b, r, cap):
    # TODO
    pass

def min_ops_varcost(customers, b, costs):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính lợi nhuận từng lượt. Mức 2: lợi nhuận phải > 0. Mức 3: (a) `4−3 = 1 > 0` ⇒ 1; (b) `0` ⇒ −1; (c) lượt 1, 2 lỗ `−2, −4`, lượt 3: `12 − 6 = 6`, lượt 4: `24 − 8 = 16` ⇒ 4; (d) lượt 1: `40 − 9 = 31`, lượt 2: `50 − 18 = 32` ⇒ 2.

**Bài 2.** Mức 1: lượt 1: đến 5, lên 4, chờ 1. Mức 2: lượt 2: đến 5, chờ 6, lên 4, chờ 2; lượt 3: lên 2. Mức 3: lợi nhuận `12−5 = 7`, `24−10 = 14`, `30−15 = 15` ⇒ 3.

**Bài 3.** Mức 1: giống Cách 2. Mức 2: không cần `ans`. Mức 3: `best` khởi tạo 0 nên tự động trả 0 khi không có lời.

**Bài 4.** Mức 1: thay mọi số 4 bằng `cap`. Mức 2: kể cả trong tua nhanh (`divmod(waiting, cap)`, `cap*b - r`). Mức 3: đối chiếu với brute force có `cap`.

**Bài 5.** Mức 1: chi phí tích lũy `spent += costs[(turns-1) % m]`. Mức 2: lợi nhuận = `boarded*b − spent`. Mức 3: không tua nhanh được đơn giản vì chi phí thay đổi — mô phỏng từng lượt.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
b1 = [Solution().minOperationsMaxProfit(*a) for a in (([4], 1, 3), ([4], 1, 4), ([0, 0, 8], 3, 2), ([5], 10, 9))]
print("Bài 1:", b1)
assert b1 == [1, -1, 4, 2]
assert trace_simulate([5, 5], 3, 5) == 3

Bài 1: [1, -1, 4, 2]
lượt | đến | chờ trước | lên | chờ sau | đã lên | lợi nhuận | best/ans
   1 |    5 |         5 |   4 |       1 |      4 |         7 | 7/1
   2 |    5 |         6 |   4 |       2 |      8 |        14 | 14/2
   3 |    - |         2 |   2 |       0 |     10 |        15 | 15/3
=> kết quả 3


In [26]:
# Bài 3
def max_profit_value_sol(customers, b, r):
    n = len(customers)
    i = waiting = boarded = turns = 0
    best = 0
    while i < n or waiting > 0:
        if i < n:
            waiting += customers[i]; i += 1
        board = min(4, waiting); waiting -= board
        boarded += board; turns += 1
        best = max(best, boarded * b - turns * r)
    return best

assert [max_profit_value_sol(c, b, r) for c, b, r, _ in examples] == [37, 122, 0]
random.seed(3)
for _ in range(500):
    cus = [random.randint(0, 50) for _ in range(random.randint(1, 10))]
    b, r = random.randint(1, 100), random.randint(1, 100)
    assert max_profit_value_sol(cus, b, r) == max([0] + profits_list(cus, b, r))
print("Bài 3: OK")

Bài 3: OK


In [27]:
# Bài 4
def min_ops_cap_sol(customers, b, r, cap):
    n = len(customers); i = waiting = boarded = turns = 0; best, ans = 0, -1
    while i < n or waiting > 0:
        if i < n:
            waiting += customers[i]; i += 1
        board = min(cap, waiting); waiting -= board
        boarded += board; turns += 1
        profit = boarded * b - turns * r
        if profit > best:
            best, ans = profit, turns
    return ans

def min_ops_cap_brute(customers, b, r, cap):
    def prof(k):
        i = waiting = boarded = 0
        for _ in range(k):
            if i < len(customers): waiting += customers[i]; i += 1
            x = min(cap, waiting); waiting -= x; boarded += x
        return boarded * b - k * r
    R = len(customers) + sum(customers) // cap + 2
    best, ans = 0, -1
    for k in range(1, R + 1):
        if prof(k) > best: best, ans = prof(k), k
    return ans

assert all(min_ops_cap_sol(c, b, r, 4) == w for c, b, r, w in examples)
random.seed(4)
for _ in range(400):
    cus = [random.randint(0, 20) for _ in range(random.randint(1, 8))]
    b, r, cap = random.randint(1, 30), random.randint(1, 60), random.randint(1, 6)
    assert min_ops_cap_sol(cus, b, r, cap) == min_ops_cap_brute(cus, b, r, cap)
print("Bài 4: OK")

Bài 4: OK


In [28]:
# Bài 5
def min_ops_varcost_sol(customers, b, costs):
    n, m = len(customers), len(costs)
    i = waiting = boarded = turns = spent = 0
    best, ans = 0, -1
    while i < n or waiting > 0:
        if i < n:
            waiting += customers[i]; i += 1
        board = min(4, waiting); waiting -= board
        boarded += board; turns += 1
        spent += costs[(turns - 1) % m]
        profit = boarded * b - spent
        if profit > best:
            best, ans = profit, turns
    return ans

# với costs hằng số, phải trùng bài gốc
random.seed(5)
for _ in range(300):
    cus = [random.randint(0, 50) for _ in range(random.randint(1, 10))]
    b, r = random.randint(1, 100), random.randint(1, 100)
    assert min_ops_varcost_sol(cus, b, [r]) == Solution().minOperationsMaxProfit(cus, b, r)
print("Bài 5:", min_ops_varcost_sol([10, 9, 6], 6, [4, 30]), "(chi phí xen kẽ 4, 30)")

Bài 5: 5 (chi phí xen kẽ 4, 30)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template mô phỏng + đỉnh sớm nhất:**

```python
i = waiting = boarded = turns = 0
best, ans = 0, -1
while i < n or waiting > 0:
    if i < n: waiting += customers[i]; i += 1
    board = min(4, waiting); waiting -= board
    boarded += board; turns += 1
    profit = boarded * b - turns * r
    if profit > best: best, ans = profit, turns
return ans
```

**Checklist:**
- [ ] Vòng lặp chạy khi `i < n` **hoặc** `waiting > 0`?
- [ ] Vẫn quay ở các lượt không có ai (khi khách còn đến sau)?
- [ ] `board = min(4, waiting)`?
- [ ] So sánh `profit > best` (nghiêm ngặt) để lấy số lượt ít nhất?
- [ ] `best = 0` ban đầu ⇒ lợi nhuận 0 không được tính, trả `-1` đúng?
- [ ] Đã thử: hòa lợi nhuận, lợi nhuận đúng 0, toàn 0 khách, `4b ≤ r`, input lớn `n = 10⁵`?